# Model Selection
This notebook is devoted to tuning the hyperparameters for our XGBoost model.

In [3]:
import numpy as np
import pandas as pd
import xgboost as xgb
import optuna
from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold, cross_val_score, cross_validate
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

In [4]:
data = pd.read_parquet(r"..\Data\sale_and_crime_data\chicago_sales_with_crime.parquet")

In [5]:
features_to_exclude = ["pin","is_multisale",'flood_fema_sfha', "sale_filter_same_sale_within_365", "sale_filter_less_than_10k", "single_v_multi_family", "zip_code",
                      "longitude", "latitude", "sale_date_parsed", "class", 'centroid_x_crs_3435', 'centroid_y_crs_3435', 'pin10', 'nearest_metra_route_dist_ft', 'nearest_new_construction_pin10',
                       "school_elementary_district_name",'nearest_golf_course_dist_ft', "school_secondary_district_name", "township_code",  "neighborhood_code",'sale_filter_deed_type', 'nearest_cta_route_dist_ft',
                                 'nearest_vacant_land_pin10','airport_noise_dnl','attic_finish','census_acs5_tract_geoid', 'nearest_neighbor_1_pin10','nearest_neighbor_2_pin10','nearest_neighbor_3_pin10','sale_date_parsed','sale_date', 'sale_year']
data = data.drop(columns = features_to_exclude)

In [6]:
cat_cols = ["type_of_residence", "construction_quality", "garage_size",
            "basement_type", "ext_wall_material", "repair_condition", "basement_finish",
            "central_heating", "central_air", "community_area"]

for col in cat_cols:
    data[col] = data[col].astype("category")

In [7]:
y=data["sale_price"]
features = list(data.columns)
features.remove("sale_price")
X=data[features]

In [9]:
# --- Same train/test split as before (test set stays untouched until final eval) ---
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.15, random_state=42
)

def objective(trial):
    params = {
        "max_depth": trial.suggest_int("max_depth", 3, 9),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "min_child_weight": trial.suggest_float("min_child_weight", 1, 20, log=True),
        "gamma": trial.suggest_float("gamma", 0, 5),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
    }

    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    rmses = []

    for train_idx, val_idx in kf.split(X_train_full):
        X_tr, X_val = X_train_full.iloc[train_idx], X_train_full.iloc[val_idx]
        y_tr, y_val = y_train_full.iloc[train_idx], y_train_full.iloc[val_idx]

        model = xgb.XGBRegressor(
            n_estimators=2000,
            tree_method="hist",
            enable_categorical=True,
            early_stopping_rounds=50,
            eval_metric="rmse",
            random_state=42,
            **params,
        )
        model.fit(
            X_tr, y_tr,
            eval_set=[(X_val, y_val)],
            verbose=False,
        )
        preds = model.predict(X_val)
        rmses.append(np.sqrt(mean_squared_error(y_val, preds)))

    return np.mean(rmses)

study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=100, show_progress_bar=True)

print("Best CV RMSE:", study.best_value)
print("Best params:", study.best_params)

[I 2026-07-09 20:29:56,308] A new study created in memory with name: no-name-d4e4d9b5-efc3-42e5-8059-9ba81862eb56


  0%|          | 0/100 [00:00<?, ?it/s]

[I 2026-07-09 20:30:33,158] Trial 0 finished with value: 170929.86072859616 and parameters: {'max_depth': 5, 'learning_rate': 0.17254716573280354, 'min_child_weight': 8.960785365368121, 'gamma': 2.993292420985183, 'subsample': 0.5780093202212182, 'colsample_bytree': 0.5779972601681014}. Best is trial 0 with value: 170929.86072859616.
[I 2026-07-09 20:31:43,004] Trial 1 finished with value: 172225.15395861017 and parameters: {'max_depth': 3, 'learning_rate': 0.13394334706750485, 'min_child_weight': 6.054365855469246, 'gamma': 3.540362888980227, 'subsample': 0.5102922471479012, 'colsample_bytree': 0.9849549260809971}. Best is trial 0 with value: 170929.86072859616.
[I 2026-07-09 20:34:17,725] Trial 2 finished with value: 163871.2434716345 and parameters: {'max_depth': 8, 'learning_rate': 0.018891200276189388, 'min_child_weight': 1.7240892195821529, 'gamma': 0.9170225492671691, 'subsample': 0.6521211214797689, 'colsample_bytree': 0.762378215816119}. Best is trial 2 with value: 163871.2434

In [11]:
import json

hyperparams = study.best_params

with open("hyperparameters.json", "w") as f:
    json.dump(hyperparams, f, indent=4)

In [17]:
# --- Refit with best params, using early stopping on a single dev split to lock in n_estimators ---
best_params = study.best_params

X_train, X_dev, y_train, y_dev = train_test_split(
    X_train_full, y_train_full, test_size=0.1765, random_state=42
)

final_model = xgb.XGBRegressor(
    n_estimators=2000,
    tree_method="hist",
    enable_categorical=True,
    early_stopping_rounds=50,
    eval_metric="rmse",
    random_state=42,
    **best_params,
)
final_model.fit(
    X_train, y_train,
    eval_set=[(X_dev, y_dev)],
    verbose=False,
)

best_n_estimators = final_model.best_iteration
print(f"Best n_estimators: {best_n_estimators}")

# --- Evaluate on the untouched test set ---
y_train_pred = final_model.predict(X_train)
y_test_pred = final_model.predict(X_test)

print("Test R²:   {:.4f}".format(r2_score(y_test, y_test_pred)))
print("Test MAE:  {:.4f}".format(mean_absolute_error(y_test, y_test_pred)))
print("Test RMSE: {:.4f}".format(np.sqrt(mean_squared_error(y_test, y_test_pred))))
print("Train R² (for overfitting check): {:.4f}".format(r2_score(y_train, y_train_pred)))

Best n_estimators: 1996
Test R²:   0.8956
Test MAE:  85414.8281
Test RMSE: 138066.2241
Train R² (for overfitting check): 0.9502


In [14]:
hyperparams["n_estimators"]= best_n_estimators

with open("hyperparameters.json", "w") as f:
    json.dump(hyperparams, f, indent=4)